# 15. Dense Data Evaluation (Dual-Branch Network)
**Objective:** Evaluate the accuracy leap achieved purely by switching to 10ms dense overlapping windows, using our champion Dual-Branch FFT + Sparse architecture.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, DEVICE
from src.frequency import extract_fft_magnitude
from src.dictionary_freq import FrequencyDictionaryLearner, FrequencyOMPExtractor
from src.fusion_net import DualBranchEMGNet, train_dual_branch_model

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load the Massive V4 Dense Dataset

In [2]:
dense_h5_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1_Dense.h5")
with h5py.File(dense_h5_path, 'r') as f:
    X_dense = np.array(f['X'])
    y_dense = np.array(f['y']).astype(np.int64)
    reps_dense = np.array(f['reps'])

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_dense, train_reps))[0]
test_idx = np.where(np.isin(reps_dense, test_reps))[0]

print(f"Total Dense Windows: {X_dense.shape[0]}")

Total Dense Windows: 186753


### 2. Extract FFT Frequency Features (Branch 1)

In [3]:
X_freq_dense = extract_fft_magnitude(X_dense)

X_freq_train, y_train = X_freq_dense[train_idx], y_dense[train_idx]
X_freq_test, y_test = X_freq_dense[test_idx], y_dense[test_idx]

print(f"Dense FFT Training Shape: {X_freq_train.shape}")
print(f"Dense FFT Testing Shape: {X_freq_test.shape}")

Extracting FFT from input shape (186753, 20, 10)...
FFT extraction complete. Output shape: (186753, 10, 10)
Dense FFT Training Shape: (126940, 10, 10)
Dense FFT Testing Shape: (54578, 10, 10)


### 3. Train Dense Dictionary & Extract Sparse Codes (Branch 2)
We must train a new dictionary so it can learn from the 100,000+ new phase-shifted representations.

In [4]:
dense_learner = FrequencyDictionaryLearner(n_atoms=32, transform_n_nonzero_coefs=3, method='minibatch')
dense_learner.fit(X_freq_train)
dense_learner.save_dictionary("emg_dict_minibatch_dense_S1.npz")

dense_extractor = FrequencyOMPExtractor(dense_learner.dictionary_, n_nonzero_coefs=3)

print("\nExtracting Sparse Codes for Dense Train Set...")
X_train_sparse_raw = dense_extractor.transform(X_freq_train)

print("Extracting Sparse Codes for Dense Test Set...")
X_test_sparse_raw = dense_extractor.transform(X_freq_test)

scaler = StandardScaler()
X_sparse_train = scaler.fit_transform(X_train_sparse_raw)
X_sparse_test = scaler.transform(X_test_sparse_raw)

print(f"Dense Sparse Features Shape: {X_sparse_train.shape}")

Training MINIBATCH Dictionary with 32 atoms on 1269400 frequency spectra...
Dictionary learning complete.
Dictionary saved to /workspaces/TCC/models/emg_dict_minibatch_dense_S1.npz

Extracting Sparse Codes for Dense Train Set...


/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


Extracting Sparse Codes for Dense Test Set...


/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


Dense Sparse Features Shape: (126940, 320)


### 4. Initialize and Train the Dual-Branch Network

In [5]:
num_channels = X_freq_train.shape[2]
freq_bins = X_freq_train.shape[1]
sparse_dim = X_sparse_train.shape[1]
num_classes = int(max(y_train.max(), y_test.max()) + 1)

dual_model = DualBranchEMGNet(
    num_channels=num_channels,
    freq_bins=freq_bins,
    sparse_dim=sparse_dim,
    num_classes=num_classes
)

dual_model = train_dual_branch_model(
    model=dual_model,
    X_freq_train=X_freq_train,
    X_sparse_train=X_sparse_train,
    y_train=y_train,
    epochs=40,
    batch_size=256, # Increased batch size drastically to handle 100k+ samples efficiently
    lr=0.001
)

Training Dual-Branch Network on cuda...
Epoch 1/40 | Loss: 2.2236
Epoch 5/40 | Loss: 1.0997
Epoch 10/40 | Loss: 0.9156
Epoch 15/40 | Loss: 0.8312
Epoch 20/40 | Loss: 0.7699
Epoch 25/40 | Loss: 0.7306
Epoch 30/40 | Loss: 0.7031
Epoch 35/40 | Loss: 0.6803
Epoch 40/40 | Loss: 0.6665


### 5. Evaluate Network

In [6]:
dual_model.eval()
with torch.no_grad():
    # Process test set in batches if memory is an issue, but 50k samples should fit on most GPUs
    X_f_te = torch.FloatTensor(X_freq_test).transpose(1, 2).to(DEVICE)
    X_s_te = torch.FloatTensor(X_sparse_test).to(DEVICE)
    
    raw_outputs = dual_model(X_f_te, X_s_te)
    dual_preds = torch.argmax(raw_outputs, dim=1).cpu().numpy()

dense_acc = accuracy_score(y_test, dual_preds)
dense_f1 = f1_score(y_test, dual_preds, average='macro')

print("\n--- Dense Dual-Branch Fusion Results ---")
print(f"Accuracy: {dense_acc * 100:.2f}%")
print(f"Macro F1-Score: {dense_f1 * 100:.2f}%")


--- Dense Dual-Branch Fusion Results ---
Accuracy: 54.37%
Macro F1-Score: 53.38%
